Airport random data

In [1]:
import random
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import os
import speech_recognition as sr  
from gtts import gTTS
from pydub import AudioSegment
from pydub.playback import play
import io



c:\Users\netra\AppData\Local\Programs\Python\Python39\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
airlines = {
    "Air Canada": "AC",
    "WestJet": "WS",
    "Flair": "Y9",
    "Lynx": "YX",
    "Porter Airlines": "PD",
    "Air Transat": "TS",
    "Sunwing Airlines": "WG"
}




# List of Canadian cities
cities= ['Toronto', 'Vancouver', 'Edmonton', 'Ottawa', 'Montreal', 'Halifax', 'Winnipeg', 'Saskatoon', 'Regina', 'Quebec City', 'Fredericton', 'St. John\'s', 'Charlottetown', 'Yellowknife', 'Iqaluit', 'Whitehorse', 'Rimouski', 'Moncton', 'Kamloops', 'Chicoutimi', 'Saguenay', 'Sherbrooke', 'Trois-Rivières', 'Gatineau', 'Brossard', 'Longueuil', 'Laval', 'Markham', 'Mississauga', 'North York', 'Scarborough', 'Thornhill', 'Vaughan', 'Ajax', 'Brampton', 'Burlington', 'Cambridge', 'Etobicoke', 'Guelph', 'Hamilton', 'Kitchener', 'London', 'Markham', 'Milton', 'Niagara Falls', 'Oakville', 'Pickering', 'Richmond Hill', 'Saint Catharines', 'St. Catharines', 'St. John']

In [3]:

def get_random_airline(airlines):
    airline_name = random.choice(list(airlines.keys()))
    airline_code = airlines[airline_name]
    return airline_name, airline_code

def generate_random_flight_number():
    return f"FL{random.randint(1000, 9999)}"  # Example: AC followed by a random 4-digit number


def get_random_city(city):
    random_city = random.choice(city)
    return random_city


def generate_random_departure_time():
    # Generate random hours, minutes, and seconds
    hour = random.randint(0, 23)  # 24-hour format
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    
    # Format the time as a string
    departure_time = f"{hour:02d}:{minute:02d}:{second:02d}"
    
    return departure_time



def generate_random_gate():
    # List of possible gate numbers or names
    gates = [
        "A1", "A2", "A3", "A4", "A5", "A6",
    "B1", "B2", "B3", "B4", "B5", "B6",
    "C1", "C2", "C3", "C4", "C5", "C6",
    "D1", "D2", "D3", "D4", "D5", "D6",
    "E1", "E2", "E3", "E4", "E5", "E6"
    ]

    # Generate a random gate by choosing from the list
    random_gate = random.choice(gates)

    return random_gate

In [4]:
flight_data = []
unique_flight_numbers = set()
used_flight_numbers = set()

for i in range(270):
    airline_name, airline_code = get_random_airline(airlines)
    city = get_random_city(cities)
    departure_time = generate_random_departure_time()
    gate = generate_random_gate()
    flight_number = generate_random_flight_number()

    # Check for overlapping departure times and gate conflicts
    overlapping = any(row[4] == departure_time for row in flight_data)
    gate_conflict = any(row[5] == gate and row[4] == departure_time for row in flight_data)

    # Check for conflicting flight numbers
    while flight_number in used_flight_numbers:
        flight_number = generate_random_flight_number()

    # Add the flight number to the set of used flight numbers
    used_flight_numbers.add(flight_number)
    
    # If there's an overlap, generate a new departure time or gate
    while overlapping or gate_conflict:
        if overlapping:
            departure_time = generate_random_departure_time()
        if gate_conflict:
            gate = generate_random_gate()
        overlapping = any(row[4] == departure_time for row in flight_data)
        gate_conflict = any(row[5] == gate and row[4] == departure_time for row in flight_data)

    # Append the flight details to the list, including the correct city name
    flight_data.append([flight_number, airline_name, airline_code, city, departure_time, gate])

# Create a Pandas DataFrame from the flight data

flight_df = pd.DataFrame(flight_data, columns=["Flight Number", "Airline Name", "Airline Code", "City", "Departure Time", "Gate"])

# Reorder the columns
flight_df = flight_df[["Flight Number", "Airline Name", "Airline Code", "City", "Departure Time", "Gate"]]

# Print the DataFrame
print(flight_df)

flight_df.to_excel("flight_details.xlsx", index=False)
















    Flight Number      Airline Name Airline Code           City  \
0          FL5683           WestJet           WS        Halifax   
1          FL2523       Air Transat           TS      Saskatoon   
2          FL7850              Lynx           YX  Niagara Falls   
3          FL2649   Porter Airlines           PD       Kamloops   
4          FL9085  Sunwing Airlines           WG       Gatineau   
..            ...               ...          ...            ...   
265        FL3296        Air Canada           AC        Halifax   
266        FL7213   Porter Airlines           PD        Halifax   
267        FL8967             Flair           Y9        Markham   
268        FL8914  Sunwing Airlines           WG       Saguenay   
269        FL1172              Lynx           YX     North York   

    Departure Time Gate  
0         23:42:03   B6  
1         10:37:42   D3  
2         07:50:36   C6  
3         02:34:26   E3  
4         00:42:07   A2  
..             ...  ...  
265       11:

In [5]:
# Download NLTK data (if you haven't already)
nltk.download('punkt')
nltk.download('stopwords')

# Load the flight data from the Excel file
flight_df = pd.read_excel("flight_details.xlsx")

# Tokenize the flight numbers and remove stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\netra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\netra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
def preprocess_input(input_text):
    print("Input Text:", input_text)
    words = word_tokenize(input_text)
    print("Tokenized Words:", words)
    filtered_words = [word for word in words if word.lower() not in stop_words]
    print("Filtered Words:", filtered_words)
    return filtered_words

# Function to search for flight details based on flight number
def search_flight_by_number(input_flight_number):
    input_flight_number = input_flight_number.upper()  # Ensure it's in uppercase
    results = flight_df[flight_df['Flight Number'] == input_flight_number]
    return results

# Function to search for flight details based on airline name
#def search_flight_by_airline(input_airline_name):
    #input_airline_name = input_airline_name.upper()
    #results = flight_df[flight_df['Airline Name'] == input_airline_name]
    #return results

def search_flight_by_airline(input_airline_name):
    input_airline_name = input_airline_name.upper()
    print("Searching for:", input_airline_name)
    results = flight_df[flight_df['Airline Name'].str.contains(input_airline_name)]
    print("Results:", results)
    return results

# Function to search for flight details based on airline code
def search_flight_by_airline_code(airline_code):
    airline_code = airline_code.upper()
    results = flight_df[flight_df['Airline Code'] == airline_code]
    return results

# Function to search for flight details based on departure time
def search_flight_by_departure_time(departure_time):
    results = flight_df[flight_df['Departure Time'] == departure_time]
    return results

# Function to search for flight details based on departure city
def search_flight_by_departure_city(departure_city):
    departure_city = departure_city.title()  # Ensure it's title case
    results = flight_df[flight_df['City'] == departure_city]
    return results

def speak(text):
    tts = gTTS(text, lang='en')
    audio_data = io.BytesIO()
    tts.write_to_fp(audio_data)
    audio_data.seek(0)
    audio = AudioSegment.from_file(audio_data)
    play(audio)
    
def clear_console():
    os.system('cls' if os.name == 'nt' else 'clear')


In [7]:
while True:
    clear_console()  # Clear the console before displaying flight information
    user_input = input("Enter a flight number, airline name, airline code, departure time, or departure city, or 'exit' to quit: ")
    input_tokens = preprocess_input(user_input)
    print(f"Input Tokens: {input_tokens}")
    print(f"User Input: {user_input}")
    
    if user_input.lower() == 'exit':
        break

    input_tokens = preprocess_input(user_input)
    flight_details = pd.DataFrame()  # Initialize flight_details with an empty DataFrame

    if any(row in input_tokens for row in flight_df['Flight Number']):
        print("Searching by Flight Number...")
        flight_details = search_flight_by_number(user_input)
    elif any(row in input_tokens for row in flight_df['Airline Name']):
        input_airline = user_input.upper()  # Convert the user input to uppercase
        print("Searching by Airline Name...")
        flight_details = search_flight_by_airline(input_airline)  # Use the uppercase input
    elif any(row in input_tokens for row in flight_df['Airline Code']):
        input_airline_code = user_input.upper()  # Convert the user input to uppercase
        print("Searching by Airline Code...")
        flight_details = search_flight_by_airline_code(input_airline_code)  # Use the uppercase input
    elif any(row in input_tokens for row in flight_df['Departure Time']):
        print("Searching by Departure Time...")
        flight_details = search_flight_by_departure_time(user_input)
    elif any(row in input_tokens for row in flight_df['City']):
        print("Searching by Departure City...")
        flight_details = search_flight_by_departure_city(user_input)

    if not flight_details.empty:
        # Display only the relevant information
        relevant_columns = ["Flight Number", "Airline Name", "Airline Code", "City", "Departure Time", "Gate"]
        relevant_flight_details = flight_details[relevant_columns]
        print(relevant_flight_details)
    else:
        print("Flight not found.")

    input("Press Enter to continue...")  # Wait for the user to press Enter




Input Text: 
Tokenized Words: []
Filtered Words: []
Input Tokens: []
User Input: 
Input Text: 
Tokenized Words: []
Filtered Words: []
Flight not found.
Input Text: 
Tokenized Words: []
Filtered Words: []
Input Tokens: []
User Input: 
Input Text: 
Tokenized Words: []
Filtered Words: []
Flight not found.
Input Text: 
Tokenized Words: []
Filtered Words: []
Input Tokens: []
User Input: 
Input Text: 
Tokenized Words: []
Filtered Words: []
Flight not found.
Input Text: escape
Tokenized Words: ['escape']
Filtered Words: ['escape']
Input Tokens: ['escape']
User Input: escape
Input Text: escape
Tokenized Words: ['escape']
Filtered Words: ['escape']
Flight not found.
Input Text: 
Tokenized Words: []
Filtered Words: []
Input Tokens: []
User Input: 
Input Text: 
Tokenized Words: []
Filtered Words: []
Flight not found.
Input Text: escape
Tokenized Words: ['escape']
Filtered Words: ['escape']
Input Tokens: ['escape']
User Input: escape
Input Text: escape
Tokenized Words: ['escape']
Filtered Words: [

In [16]:

recognizer = sr.Recognizer()
flight_details = pd.DataFrame() 
while True:
    clear_console()  # Clear the console before displaying flight information

    try:
        '''''
        with sr.Microphone() as source:
            recognizer.adjust_for_ambient_noise(source)  # Adjust for ambient noise
            print("Speak now...")
            audio = recognizer.listen(source, timeout=5)  # Listen for a maximum of 5 seconds
            print("Recognizing...")
            user_input = recognizer.recognize_google(audio, show_all=True)  # Use Google Web Speech API for recognition

        if user_input:
            user_input_text = user_input.get("alternative")[0].get("transcript")
            
            # Check if the input looks like a flight number (e.g., "FL 4781")
            if any(char.isdigit() for char in user_input_text):
                user_input_text = user_input_text.replace(" ", "")  # Remove spaces
            else:
                # Keep spaces for non-flight number inputs
                print(f"Recognized: {user_input_text}")

            user_input_text = user_input_text.upper()  # Convert recognized text to uppercase
            input_tokens = preprocess_input(user_input_text)
        '''''
        print("Enter a flight number, airline name, airline code, departure time, or departure city, or 'exit' to quit:")
        user_input = input("")
        speak("You entered: " + user_input)  # Provide audio feedback for the input

        if user_input.lower() == 'exit':
            break

        input_tokens = preprocess_input(user_input)
        flight_details = pd.DataFrame()  # Initialize flight_details with an empty DataFrame


        if any(row in input_tokens for row in flight_df['Flight Number']):
            flight_details = search_flight_by_number(user_input)
        elif any(row in input_tokens for row in flight_df['Airline Name']):
            flight_details = search_flight_by_airline(user_input)  # Changed the search function to match airline name
        elif any(row in input_tokens for row in flight_df['Airline Code']):
            flight_details = search_flight_by_airline_code(user_input)
        elif any(row in input_tokens for row in flight_df['City']):
            flight_details = search_flight_by_departure_city(user_input)

        if not flight_details.empty:
            # Display only the relevant information
            relevant_columns = ["Flight Number", "Airline Name", "Airline Code", "City", "Departure Time", "Gate"]
            relevant_flight_details = flight_details[relevant_columns]
            print(relevant_flight_details)
        else:
            speak("Flight not found.")

    except sr.UnknownValueError:
        print("Sorry, I could not understand what you said.")
    except sr.RequestError:
        print("Sorry, I encountered an error while trying to recognize your speech.")
    except Exception as e:
        print(f"An error occurred: {e}")

    speak("Press Enter to continue...")  # Provide audio feedback for continuation
    input("Press Enter to continue...")  # Wait for the user to press Enter







Enter a flight number, airline name, airline code, departure time, or departure city, or 'exit' to quit:


c:\Users\netra\AppData\Local\Programs\Python\Python39\lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
c:\Users\netra\AppData\Local\Programs\Python\Python39\lib\site-packages\pydub\utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
  warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)


An error occurred: [WinError 2] The system cannot find the file specified


FileNotFoundError: [WinError 2] The system cannot find the file specified